# ConnectomeBench: Controles e Linhas de Base (Baselines)

**Fase 4 do Projeto**

Neste caderno validamos nossas "Redes Falsas" que servirão como competidoras para o FlyWire.
A Ciência exige que comparemos a topologia biológica contra controles justos para afirmar que existe vantagem.

Controles:
1. **Rede Aleatória (Random Sparse):** Mesma densidade de conexões, mas alocadas aleatoriamente.
2. **Rede Clonada (Degree-Matched Sparse):** Preserva a quantidade de conexões exata de cada nó (hubs continuam hubs), mas embaralha *quem* se conecta com quem.


In [ ]:
import sys
import os
import time

# Adiciona o diretório raiz ao path
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Componentes do ConnectomeBench
from src.graph.builder import ConnectomeBuilder, NodeMapper
from src.topology.metrics import get_degrees, calculate_sparsity
from src.models.baselines import generate_random_sparse, generate_degree_matched

# Configurações de plotagem
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)


## 1. Carregando a Rede Biológica Real


In [ ]:
CONNECTIONS_PATH = "../data/raw/connections_princeton.csv.gz"

print("1. Lendo mapeamento e criando Grafo Base (FlyWire)...")
import pandas as pd
df = pd.read_csv(CONNECTIONS_PATH)
builder = ConnectomeBuilder(df)



# Usaremos uma versão puramente estrutural (binária) para métricas de grau
adj_real, mapper = builder.build_sparse_matrix(weight_col='syn_count')
adj_real.data = np.ones_like(adj_real.data)

real_density = 1.0 - calculate_sparsity(adj_real)
real_in_deg, real_out_deg = get_degrees(adj_real)

print(f"Nodos: {mapper.num_nodes:,}")
print(f"Arestas (Únicas): {adj_real.nnz:,}")
print(f"Densidade: {real_density:.6%}")


## 2. Gerando a Rede Aleatória (Random Sparse)
Tem a mesma densidade, mas nenhuma organização biológica.


In [ ]:
print("2. Gerando Rede Aleatória...")
start = time.time()
adj_random = generate_random_sparse(mapper.num_nodes, density=real_density, seed=42)
end = time.time()

print(f"Tempo de geração: {end - start:.2f}s")
print(f"Arestas criadas: {adj_random.nnz:,}")

rand_in_deg, rand_out_deg = get_degrees(adj_random)


## 3. Gerando a Rede Clonada (Degree-Matched)
Tem a mesma popularidade por nó, mas as amizades são aleatórias.
Usamos o algoritmo ultra-rápido de Matching de Stubs.


In [ ]:
print("3. Gerando Rede Clonada (Degree-Matched)...")
start = time.time()
adj_matched = generate_degree_matched(real_in_deg, real_out_deg, seed=42)
end = time.time()

print(f"Tempo de geração: {end - start:.2f}s")
print(f"Arestas criadas: {adj_matched.nnz:,}")

# É normal que a rede matched tenha levemente menos arestas que o original 
# porque o modelo colide múltiplas arestas ou se auto-conecta, e nós binarizamos.
matched_in_deg, matched_out_deg = get_degrees(adj_matched)


## 4. Comparação de Distribuições (Prova Científica)
A Prova dos Nove: A distribuição da Rede Aleatória será uma "sino" normal. A Rede Clonada deve ser um **espelho exato** do Cérebro da Mosca.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Filtra apenas os que têm > 0 para visualização logarítmica limpa
def plot_hist(ax, deg_array, label, color):
    sns.kdeplot(deg_array[deg_array > 0], log_scale=True, ax=ax, label=label, color=color, fill=True, alpha=0.3)

# Comparação Out-Degree
plot_hist(axes[0], real_out_deg, "Cérebro (Real)", "blue")
plot_hist(axes[0], rand_out_deg, "Aleatória", "red")
plot_hist(axes[0], matched_out_deg, "Clonada", "green")
axes[0].set_title('Densidade de Out-Degree (Transmissão)')
axes[0].set_xlabel('Out-degree (escala log)')
axes[0].legend()

# Comparação In-Degree
plot_hist(axes[1], real_in_deg, "Cérebro (Real)", "blue")
plot_hist(axes[1], rand_in_deg, "Aleatória", "red")
plot_hist(axes[1], matched_in_deg, "Clonada", "green")
axes[1].set_title('Densidade de In-Degree (Recepção)')
axes[1].set_xlabel('In-degree (escala log)')
axes[1].legend()

plt.tight_layout()
plt.show()


## Conclusão
- A **Rede Aleatória** falha catastroficamente em capturar a natureza "Scale-Free" (Cauda-longa) do cérebro.
- A **Rede Clonada** sobrepõe perfeitamente o cérebro original.

Quando criarmos modelos de Inteligência Artificial usando `adj_matched` em vez de `adj_real`, descobriremos se o ganho de eficiência da biologia vem de **ter grandes hubs (o que a Clonada tem)** ou da **especificidade exata das conexões (o que apenas a biologia tem)**.
